# Example of Exploratory Data Analysis and ML Modeling

In this notebook, we will walk through the key stages of a Data Science workflow using a housing dataset from King County. You have already seen some of the early steps in a previous lecture. This time, we will go further by building and evaluating machine learning models.

As a quick reminder, here is the Data Science lifecycle:

<center><img src="./assets/DS_LC.png" alt="drawing" width="500"/></center>

The goal is to implement the essential steps of a supervised machine learning workflow with the Python library scikit-learn.  

We will use the following methods:  
- feature engineering,
- train/test split,
- model training,
- hyperparameter optimization with grid search and cross-validation,
- and model evaluation.

To make the workflow more concrete, we will use the following fictional scenario:

**You have just landed your dream job as a data scientist at an innovative company near Seattle. Before you move, you want to understand the local housing market and estimate fair property values so you can negotiate with confidence.**

```mermaid
flowchart LR
    A[Raw housing data] --> B[Data preparation]
    B --> C[Exploratory data analysis]
    C --> D[Feature engineering]
    D --> E[Model training]
    E --> F[Evaluation and tuning]
    F --> G[Save the final model]
```


> **This is the refactored copy of the notebook.**
> The data cleaning and feature engineering that used to live in cells now live in the
> tested `kc_house` package (`src/kc_house/`). This notebook imports it, keeps the
> exploratory analysis and the modeling story, and shows how the pipeline is used.
> The original, untouched notebook is [King-County.ipynb](King-County.ipynb).
>
> Run it with `uv sync --all-extras` and select the `.venv` kernel.

![](assets/seattle_skyline.jpg)
(<span>Photo by <a href="https://unsplash.com/@phoebezzf?utm_source=unsplash&amp;utm_medium=referral&amp;utm_content=creditCopyText">Zhifei Zhou</a> on <a href="https://unsplash.com/s/photos/seattle-skyline?utm_source=unsplash&amp;utm_medium=referral&amp;utm_content=creditCopyText">Unsplash</a></span>)

**Rather than relying on intuition alone, we will use data to support the house-hunting process. Our aim is to predict home prices in and around Seattle so that we can estimate a realistic property value before negotiating.**

## Business Understanding

Seattle is located in King County, which is shown on the map below. To keep commuting time manageable, we limit the property search to this area.



![](assets/king_county_districts.jpeg)

(Photo from King County [imap](https://gismaps.kingcounty.gov/iMap/))

For this project, we use a dataset that contains property sales in King County from May 2014 to May 2015.

Before building any model, it is important to understand what information the dataset contains. The table below summarizes the available columns:

| **Column name** | **Description** |
|---|---|
| `id` | Unique identifier for a house sale |
| `date` | Date of sale |
| `price` | Sale price of the house |
| `bedrooms` | Number of bedrooms |
| `bathrooms` | Number of bathrooms; `.5` indicates a toilet room without a shower |
| `sqft_living` | Interior living area in square feet |
| `sqft_lot` | Lot size in square feet |
| `floors` | Number of floors |
| `waterfront` | Whether the house has a waterfront view (`0` = no, `1` = yes) |
| `view` | Index from `0` to `4` indicating the quality of the view |
| `condition` | Index from `1` to `5` describing the condition of the property |
| `grade` | Index from `1` to `13` describing construction and design quality |
| `sqft_basement` | Basement area in square feet |
| `yr_built` | Year the house was built |
| `yr_renovated` | Year of the last renovation |
| `zipcode` | ZIP code of the property |
| `lat` | Latitude |
| `long` | Longitude |
| `sqft_living15` | Interior living area of the nearest 15 neighbors |
| `sqft_lot15` | Lot size of the nearest 15 neighbors |



## Importing the libraries

The notebook now imports the cleaning and feature code from the package instead of defining it inline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.linear_model import LinearRegression

from kc_house.cleaning import (
    add_last_known_change,
    drop_implausible_rows,
    fix_sqft_basement,
    impute_view_and_waterfront,
)
from kc_house.data import (
    load_raw,
    make_grouped_split,
    make_temporal_split,
    make_training_split,
)
from kc_house.evaluate import error_by_price_band, regression_metrics
from kc_house.features import WaterDistance, add_center_distance
from kc_house.models import build_production_model
from kc_house.pipeline import build_model_pipeline
from kc_house.train import tune_reference_model

Load the raw data, unchanged. (The path is relative to the repository root: run the notebook from there, or open it with VS Code.)

In [ ]:
raw = load_raw()
raw.head(10)

## Data preparation

The same three problems as in the original notebook. We look at them on the **raw** data, then fix each with a small,
named, tested function.

In [ ]:
raw.info()

**1. An implausible record.** A house with 33 bedrooms and 2 bathrooms in 1,620 square feet is a data-entry error.

In [ ]:
raw.query("bedrooms == 33")

**2. A text column that should be a number.** `sqft_basement` contains `?`, so pandas reads the whole column as text.

In [ ]:
(raw["sqft_basement"] == "?").sum()

**3. Missing values** in `waterfront`, `view` and `yr_renovated`.

In [ ]:
raw.isna().sum()[lambda s: s > 0]

Each fix is one function in `src/kc_house/cleaning.py`, and none changes its input:

| Function | What it does |
|---|---|
| `drop_implausible_rows` | removes the 33-bedroom record by rule (the original dropped row `15856` by position) |
| `fix_sqft_basement` | rebuilds basement area as living area minus above-ground area |
| `impute_view_and_waterfront` | fills missing values with 0, their dominant value |
| `add_last_known_change` | renovation year if there is one, else build year |

In [ ]:
kc_data = drop_implausible_rows(raw)
kc_data = fix_sqft_basement(kc_data)
kc_data = impute_view_and_waterfront(kc_data)
kc_data = add_last_known_change(kc_data)
kc_data.shape

Nothing is missing any more, and the basement column is numeric.

In [ ]:
kc_data.isna().sum()[lambda s: s > 0], kc_data["sqft_basement"].dtype

## Exploratory Data Analysis

Now we move from cleaning to exploration.
Histograms are a useful first step because they show how each variable is distributed and whether a variable behaves more like a continuous measure or a category.



In [ ]:
# Select variables for a closer visual inspection.
columns_histogram = [
    "price",
    "bathrooms",
    "bedrooms",
    "floors",
    "grade",
    "last_known_change",
    "sqft_living",
    "sqft_lot",
]

In [ ]:
# Plot histograms.
kc_data[columns_histogram].hist(bins=50, figsize=(20, 15))
plt.show()

Histograms help us understand both the overall distribution and the rough shape of each variable.

Findings:
- `price` is right-skewed, with a small number of very expensive houses stretching the distribution.
- More than 5000 houses have 2.5 bathrooms. In the U.S., `.5` typically indicates a toilet room without a shower.
- Most houses have between 2 and 5 bedrooms, with 3 bedrooms being the most common.
- Most houses have only one floor.
- Most houses have a grade between 6 and 10 on the 1 to 13 scale.
- Very few houses were built or renovated between 1930 and 1940. Construction increases noticeably after that period.
- `sqft_living` is also somewhat right-skewed.
- Extreme lot sizes distort the lot-area histogram, which makes the main distribution harder to see.

Another useful view is the box plot, which displays the same summary statistics graphically.



In [ ]:
# Plot box plots.
fig = px.box(kc_data, y="price", labels={"price": "House Price in $"})
fig.show()

If we try the box plot with different columns, we can clearly see outliers in several variables.  

Outliers matter because they can strongly affect model quality, but they should not be removed automatically.

For the property with 33 bedrooms, the value was implausible enough to treat it as an error and remove the row.
By contrast, very expensive houses are rare but believable in the Seattle area, so we keep them.

This is a good example of why domain knowledge matters: **an unusual value is not always a wrong value**.



It is also useful to inspect relationships between variables.
A pair plot gives a quick overview by plotting each variable against the others and showing the univariate distributions on the diagonal.

When reading the pair plot, we are especially interested in possible relationships such as:
- how price changes as living area increases
- whether better grades are associated with higher prices
- whether some features appear to move together



In [ ]:
# Plot scatterplots.
sns.pairplot(kc_data[columns_histogram]);

A heatmap provides another view of correlation, this time by summarizing the linear relationship between every pair of variables.

Strong red tones indicate positive correlation, while strong blue tones indicate negative correlation. For example:
- larger living area is often associated with higher price
- greater distance from the wealth center may be associated with lower price

**Correlation does not prove causation**, but it helps us identify promising variables for a linear model.



In [ ]:
# Heatmap of the Pearson correlation coefficients
numeric_kc_data = kc_data.select_dtypes(include=["number"])
mask = np.triu(numeric_kc_data.corr())
plt.figure(figsize=(20, 15))
ax = sns.heatmap(round(numeric_kc_data.corr(), 2), annot=True, mask=mask, cmap="RdBu_r")

## Feature engineering

Feature engineering means creating new variables that may explain the target more clearly.

* `sqft_price` (price per square foot) is derived **from the target**, so it is for exploration only and is never a model input.
* `center_distance` and `water_distance` are kept as features. They live in `src/kc_house/features.py`.

`water_distance` is a fitted step (`WaterDistance`): it learns the waterfront reference points from the rows it is fitted on.
Here, for exploration, we fit it on all rows. **In the pipeline it is fitted on training rows only**, which removes the test-to-train
leakage the original notebook had.

In [ ]:
kc_data["sqft_price"] = (
    kc_data.price / (kc_data.sqft_living + kc_data.sqft_lot)
).round(2)
kc_data = add_center_distance(kc_data)
kc_data = WaterDistance().fit(kc_data).transform(kc_data)
kc_data[["center_distance", "water_distance"]].describe().round(2)

**Distance to the center of wealth**

One possible explanatory feature is the distance to a highly desirable area. As a simple proxy, we use Bill Gates' residence in Medina, on Lake Washington, as a rough "center of wealth".
According to Wikipedia, the estate has an estimated value of about **US$147.5 million**.

<center><a href="https://en.wikipedia.org/wiki/Bill_Gates%27s_house#/media/File:Residence_of_Bill_Gates.jpg"><img src="https://upload.wikimedia.org/wikipedia/commons/2/26/Residence_of_Bill_Gates.jpg" alt="Bill Gates residence" width="700"></a></center>

To create this feature, we compute the geographic distance between each house and that reference point.
We correct longitude by `cos(lat)` because the physical distance represented by one degree of longitude depends on latitude.
This gives us a more realistic distance in kilometers.


Now let us check whether this assumption is supported by the data: are properties closer to the wealth center generally more expensive?



In [ ]:
# Scatter plot: price versus distance to the center.
sns.relplot(y="price", x="center_distance", data=kc_data);

In [ ]:
# Scatter plot: price per square foot versus distance to the center.
sns.relplot(y="sqft_price", x="center_distance", data=kc_data);

The charts support our hypothesis: price tends to decrease as distance from the wealth center increases.
The price-performance ratio also improves with distance, because the price per square foot tends to fall.

We can also plot all houses on a map of King County.
The color indicates how close each house is to the wealth center, while the marker size reflects sale price.



In [ ]:
fig = px.scatter_map(
    kc_data,
    lat="lat",
    lon="long",
    hover_name="id",
    hover_data=["sqft_price", "sqft_living", "zipcode", "floors"],
    size="sqft_price",
    color="center_distance",
    color_continuous_scale=["green", "yellow", "red"],
    color_discrete_sequence=["fuchsia"],
    zoom=7.7,
    height=400,
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

**Distance to the waterfront**

Next, we construct a rough measure of water proximity.
We first extract all houses with `waterfront = 1` into a list called `water_list`. Then, for each house in the dataset, we calculate the minimum distance to any house in that list.

This is only an approximation, because waterfront houses do not describe the entire shoreline perfectly, but it still gives us a useful proxy for how close a property is to the water.



The new variables are appended to the dataset.
On average, the houses in our dataset are about 17 km from the wealth center and almost 19 km from a house with a waterfront view. The house farthest from the wealth center is roughly 70 km away.

Again, we can test whether proximity to the water appears to influence price.



In [ ]:
sns.relplot(y="price", x="water_distance", data=kc_data);

In [ ]:
sns.relplot(y="sqft_price", x="water_distance", data=kc_data);

These charts again support our hypothesis: price tends to decrease as distance from the water increases, and the price per square foot also falls.

The map offers an additional sanity check. Houses close to the water should have small `water_distance` values, and that is exactly what we see.



In [ ]:
fig = px.scatter_map(
    kc_data.query("water_distance <= 5"),
    lat="lat",
    lon="long",
    hover_name="id",
    hover_data=["sqft_price", "sqft_living", "zipcode", "floors"],
    size="sqft_price",
    color="water_distance",
    color_continuous_scale=["green", "yellow", "red"],
    color_discrete_sequence=["fuchsia"],
    zoom=8.4,
    height=400,
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

# Modeling

The preparation steps are now one scikit-learn `Pipeline`. Because the pipeline owns cleaning and feature engineering,
**we split the raw data first** and nothing is learned from the test rows.

```mermaid
flowchart LR
    A[Raw columns] --> B[Clean] --> C[Centre distance] --> D[Water distance<br/>fitted on train] --> E[19 features] --> F[Model]
```

`make_training_split` drops the implausible record and splits 70/30 with the same seed as the original notebook.

In [ ]:
X_train, X_test, y_train, y_test = make_training_split(raw)
print("train:", X_train.shape, " test:", X_test.shape)

### A baseline: quality grade only

Any model has to beat this simple one to earn its complexity.

In [ ]:
baseline = LinearRegression().fit(X_train[["grade"]], y_train)
print("Grade only, test R²:", round(baseline.score(X_test[["grade"]], y_test), 3))

### The original notebook's model

Second-order polynomial features and ElasticNet, now as `build_model_pipeline()`. It takes the **raw** columns.

In [ ]:
reference = build_model_pipeline().fit(X_train, y_train)
print("Notebook model, test R²:", round(reference.score(X_test, y_test), 4))

The original notebook reported 0.8443 on the same split. The tiny difference is expected: the water feature is now learned from training rows only.

### Tuning with cross-validation

The same grid as the original notebook, run on training data only.

In [ ]:
search = tune_reference_model(X_train, y_train)
print("Best settings:", search.best_params_)
print("Cross-validated R²:", round(search.best_score_, 3))

### A better model

Comparing model families under the same preprocessing (see `docs/model-card.md`) showed that gradient-boosted trees beat
the polynomial ElasticNet clearly. The settings below were found by `uv run kc-train` (a random search with cross-validation on the
training data).

In [ ]:
champion = build_production_model().set_params(
    model__learning_rate=0.05,
    model__max_iter=600,
    model__max_leaf_nodes=31,
    model__min_samples_leaf=20,
    model__l2_regularization=0.1,
    model__loss="squared_error",
)
champion.fit(X_train, y_train)
metrics = regression_metrics(y_test, champion.predict(X_test))
typical = metrics["median_abs_pct_error"]
print(f"Test R²: {metrics['r2']:.3f}   typical error: {typical:.1f}% of the price")

### Error analysis

One overall number hides who is served badly. Errors per price quartile:

In [ ]:
predictions = champion.predict(X_test)
pd.DataFrame(error_by_price_band(y_test, predictions)).round(1)

The model pulls estimates toward the middle: cheap houses are overestimated, the most expensive underestimated. Where are the big misses?

In [ ]:
df_error = pd.DataFrame(
    {
        "price": y_test.to_numpy(),
        "price_prediction": predictions.round(2),
        "latitude": X_test["lat"].to_numpy(),
        "longitude": X_test["long"].to_numpy(),
        "id": X_test["id"].to_numpy(),
    }
)
df_error["price_difference_percent"] = (
    (df_error["price_prediction"] - df_error["price"]) / df_error["price"] * 100
).round(2)
fig = px.scatter_map(
    df_error,
    lat="latitude",
    lon="longitude",
    hover_data=["price", "price_prediction", "id"],
    color="price_difference_percent",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7,
    height=400,
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

### Is the score inflated?

A random split can put two sales of the same house on both sides (177 houses were sold twice), and real use means
predicting the future from the past. Same model, stricter splits:

In [ ]:
from sklearn.base import clone

rows = []
for name, make_split in [
    ("random (original notebook)", make_training_split),
    ("grouped by house id", make_grouped_split),
    ("train past, test future", make_temporal_split),
]:
    xt, xs, yt, ys = make_split(raw)
    fitted = clone(champion).fit(xt, yt)
    rows.append({"split": name, **regression_metrics(ys, fitted.predict(xs))})
pd.DataFrame(rows).set_index("split")[["r2", "mae", "median_abs_pct_error"]].round(3)

## Conclusion

* The refactored pipeline reproduces the original notebook's numbers (see `tests/test_parity.py`) while removing its leakage.
* The notebook's model reaches an R² of about 0.84. Tuned gradient boosting reaches about 0.88 and holds up under grouped and temporal splits.
* The same code now powers training (`uv run kc-train`), the API and the web page.

Next: [docs/](docs/) for the story, `uv run kc-train` to retrain, and `docker compose up --build` to run the service.